In [ ]:
# !pip install torch torchvision scikit-learn matplotlib seaborn
import os, torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from copy import deepcopy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

In [ ]:
DATA_DIR  = "./data/OCT2017"   # train/ val/ test/ inside
CLASSES   = ["CNV","DME","DRUSEN","NORMAL"]
IMG_SIZE  = 224
BATCH     = 32
EPOCHS    = 20
LR        = 1e-3

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", train_tf)
val_ds   = datasets.ImageFolder(f"{DATA_DIR}/val",   val_tf)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/test",  val_tf)

train_dl = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
def make_resnet50():
    m = models.resnet50(weights="IMAGENET1K_V1")
    for p in m.parameters(): p.requires_grad = False       # freeze
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, 4))
    return m.to(DEVICE)

def make_mobilenetv2():
    m = models.mobilenet_v2(weights="IMAGENET1K_V1")
    for p in m.parameters(): p.requires_grad = False
    m.classifier[1] = nn.Linear(1280, 4)
    return m.to(DEVICE)

def make_efficientnet():
    m = models.efficientnet_b0(weights="IMAGENET1K_V1")
    for p in m.parameters(): p.requires_grad = False
    m.classifier[1] = nn.Linear(1280, 4)
    return m.to(DEVICE)

def make_cnn():
    def block(ci, co):
        return nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
            nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(
        block(3,32), block(32,64), block(64,128), block(128,256),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Dropout(0.4), nn.Linear(256,4)
    ).to(DEVICE)

In [ ]:
def train_one_epoch(model, dl, opt, loss_fn):
    model.train(); total, correct = 0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(x); loss = loss_fn(out, y)
        loss.backward(); opt.step()
        correct += (out.argmax(1)==y).sum().item(); total += y.size(0)
    return correct/total

@torch.no_grad()
def evaluate(model, dl, loss_fn):
    model.eval(); total, correct, running_loss = 0, 0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        running_loss += loss_fn(out, y).item()
        correct += (out.argmax(1)==y).sum().item(); total += y.size(0)
    return correct/total, running_loss/len(dl)

def train_model(model, name, epochs=EPOCHS):
    opt      = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    sched    = optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
    loss_fn  = nn.CrossEntropyLoss()
    best_acc, best_w = 0, None
    history  = {"train_acc":[], "val_acc":[], "val_loss":[]}

    for ep in range(1, epochs+1):
        tr_acc       = train_one_epoch(model, train_dl, opt, loss_fn)
        val_acc, v_l = evaluate(model, val_dl, loss_fn)
        sched.step()
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        history["val_loss"].append(v_l)
        if val_acc > best_acc:
            best_acc = val_acc; best_w = deepcopy(model.state_dict())
        print(f"[{name}] Ep {ep:02d} | TR {tr_acc:.3f} | VAL {val_acc:.3f} | Loss {v_l:.4f}")

    model.load_state_dict(best_w)
    torch.save(best_w, f"{name}_best.pth")
    return model, history

In [ ]:
models_dict = {
    "CNN"         : make_cnn(),
    "ResNet50"    : make_resnet50(),
    "MobileNetV2" : make_mobilenetv2(),
    "EfficientNet": make_efficientnet(),
}

histories = {}
for name, model in models_dict.items():
    print(f"\n{'='*40}\nTraining {name}\n{'='*40}")
    models_dict[name], histories[name] = train_model(model, name)

In [ ]:
@torch.no_grad()
def get_preds(model, dl):
    model.eval(); all_p, all_y = [], []
    for x, y in dl:
        all_p.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
        all_y.extend(y.numpy())
    return np.array(all_y), np.array(all_p)

fig, axes = plt.subplots(1, 4, figsize=(20,5))
results = {}
for i, (name, model) in enumerate(models_dict.items()):
    y_true, y_pred = get_preds(model, test_dl)
    acc = (y_true==y_pred).mean()
    results[name] = acc
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", ax=axes[i],
                xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
    axes[i].set_title(f"{name}\nAcc: {acc:.3f}")
    print(f"\n--- {name} ---\n{classification_report(y_true, y_pred, target_names=CLASSES)}")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

In [ ]:
import pandas as pd

# collect per-class F1 for each model
rows = []
for name, model in models_dict.items():
    y_true, y_pred = get_preds(model, test_dl)
    from sklearn.metrics import f1_score, accuracy_score
    row = {"Model": name, "Accuracy": accuracy_score(y_true, y_pred)}
    f1s = f1_score(y_true, y_pred, average=None)
    for c, f in zip(CLASSES, f1s): row[f"F1_{c}"] = round(f, 3)
    row["F1_Macro"] = round(f1_score(y_true, y_pred, average="macro"), 3)
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")
df["Accuracy"] = df["Accuracy"].round(4)
print(df.to_string())
df.to_csv("results_summary.csv")